#### Q1
At time t, if we choose the action with the lowest expected loss then the regret for that specific step is 0. If we choose the other arm then the expected regret is the differencd between the expected losses of the two arms, which is big delta. If we try to average out the randomness in the bernoulli losses and let $\Delta = \max_{i}E(l_{t}(i)) - \min_{i}E(l_{t}(i))$ be the big delta defined right before Q1, and $smallest=\argmin_{i}E(l_{t}(i))$ then at any given $t$
$$
\begin{align*}
E(R_t) &= p_t(I_{smallest}) \times 0 + (1-p_t(I_{smallest})) \times \Delta\\
&= (1-p_t(I_{smallest}))\Delta\\
\end{align*}
$$
We see that the expected regret at $t$ depends on the probability distribution over the choice of arms at $t$ and also $\Delta$. If we want to know about $E(R_T)$ we'd need to sum the $E(R_t)$ over all $t$ until T (since the expectation is linear): $E(R_T)=\Delta\sum_{t=1}^T(1-p_t(I_{smallest}))=\Delta\left(T-\sum_{t=1}^Tp_t(I_{smallest}))\right)$. But anyway, since this question is only about the $q$ pairs we can condition on the algorithm (imagine that the way we choose the arm at any $t$ is fixed) and we find that the bigger $\Delta$, the bigger $E(R_T)$. So given that, and since the expected loss for arm $i$ is just $q_i$, I would expect all pairs to have the same expected regret because in all of them $q_1$ and $q_2$ are $0.1$ apart.

#### Q2
Code is in Q4 with results printed out at the bottom of the cell.

#### Q3
We are not supposed to know which pair generates our losses, however we know it is either one of those two pairs for sure, and one thing they have in common is that one of the two arms will have zero variance in the losses it yields because it is deterministic ($p=0$ or $p=1$). So instead of having this long exploration phase, we can watch what happens in the first couple steps:

$t=1$:<br>
&nbsp;&nbsp; <b>For</b> $i \in [K]$<br>
&nbsp;&nbsp;&nbsp;&nbsp; Observe $l_t(i)$<br>
$t\geq2$:<br>
<b>While</b> true<br>
&nbsp;&nbsp; <b>For</b> $i \in [K]$<br>
&nbsp;&nbsp;&nbsp;&nbsp; Observe $l_t(i)$<br>
&nbsp;&nbsp;&nbsp;&nbsp; <b>If</b> $l_t(i)\neq l_{t-1}(i)$<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; <b>Return</b> $i$

If either one prints a different value from the previous round: $l_t(i)\neq l_{t-1}(i)$, then the loop above breaks and given that we know the two possible true $q$ pairs, we can find the best arm easily: we now know that the other arm whose loss did not change is the deterministic arm, and:
- if the loss of that deterministic arm is $1$ then we know it's the first pair and arm $2$ is the best ($q_2=0.9$)
- if the loss of that deterministic arm is $0$ then we know it's the second pair and arm $1$ is the best ($q_1=0$)

Finally for all the remaining rounds until $T$, we always pick that best arm.

#### Q4

In [18]:
import numpy as np

K = 2
T = 10_000
M = 20

seed = 2026
rng = np.random.default_rng(seed=seed)
pairs = [
    [.1,0]
    , [.2,.1]
    ,[.5,.4]
    ,[.4,.5]
    , [.7,.8]
    , [.9,1]
]

def sample_loss(rng, q):
    success = rng.binomial(n=1, p=q, size=1).item()
    return 1 if success else 0

In [19]:
def eps_greedy(q1, q2, eps, rng):
    smallest = np.argmin([q1,q2])
    n = np.zeros(K)
    cumul_losses = np.zeros(K)
    epsT = int(np.ceil(eps*T))
    regret = 0
    sampled_losses=[] #useless but to remove annoying unbound warning

    for t in range(epsT):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        I_t = np.argmin(n)
        n[I_t] += 1
        cumul_losses[I_t] += sampled_losses[I_t]
        regret += sampled_losses[I_t] - sampled_losses[smallest]
    best = np.argmin(cumul_losses)
    for t in range(epsT, T):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        regret += sampled_losses[best] - sampled_losses[smallest]

    return regret, epsT


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}} {'epsT':>{4}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    q1=pair[0]
    q2=pair[1]
    delta = np.max([q1,q2])-np.min([q1,q2]) #expected loss is just q
    eps = 8*np.log(T*delta**2/4)/(T*delta**2)   
    for i in range(M):
        regrets[i], epsT = eps_greedy(
            q1
            , q2
            , eps
            , rng=rng
        )

    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    epsT = int(np.ceil(eps*T))
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f} {epsT:>{4}}",flush=True)

print(f"Summed average regret: {summed_avg_regret:<{8}.3f}\nStandard error: {np.sqrt(var_summed_regret):<{7}.3f}",flush=True)

pair              avg regret         sd epsT
[0.1, 0]             129.500     12.258 2576
[0.2, 0.1]           125.150     18.027 2576
[0.5, 0.4]           130.350     27.225 2576
[0.4, 0.5]           129.150     22.041 2576
[0.7, 0.8]           130.100     21.762 2576
[0.9, 1]             132.700      9.257 2576
Summed average regret: 776.950 
Standard error: 47.555 


Indeed, our expectations from Q1 seem to be verified in this sample, looking at the average regrets. The number of explorations steps is the same for all ($2576$) since $\Delta$ is the same for all pairs. Since during the exploration phase we sample equally from both arms, each arm is sampled $1288$ times in this phase, so we will necessarily make at least $1288$ mistakes, and since $\Delta$ is $0.1$ for all pairs, we can expect the regret at $T$ to be at least $128.8$ on average. In practice might be less if during the exploration phase the empirical averages of the bernoulli losses are closerb than their theoretical averages are, and it might more if the opposite happened or (very low probability with that epsilon) we ended up choosing the wrong arm after exploration. The standard errors follow a pyramid-like structure, that is expected because the variance of bernoulli losses is larger the closer $q_i$ is to $.5$, and smaller the further it is from $.5$.

In [ ]:
def lcb1(q1, q2, delta, rng):
    smallest = np.argmin([q1,q2])

    #first we init the variables (t=1 to t=K)
    cumul_losses = np.zeros(K)
    n = np.repeat(1,K)
    regret = 0
    for t in range(K):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        regret += sampled_losses[t] - sampled_losses[smallest]
        cumul_losses[t] += sampled_losses[t]
    mu_hats = np.copy(cumul_losses)
    mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta))

    for t in range(K, T):
        I_t = np.argmin(mu_deltas)

        #update variables for next iter:
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[I_t] += sampled_losses[I_t]

        n[I_t] += 1
        mu_hats = cumul_losses/n
        mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta)/n)

        regret += sampled_losses[I_t] - sampled_losses[smallest]

    return regret


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    for i in range(M):
        regrets[i] = lcb1(
            q1=pair[0]
            , q2=pair[1]
            , delta=1/T**2
            , rng=rng
        )
    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f}",flush=True)

print(f"{summed_avg_regret:>{14}.3f} {np.sqrt(var_summed_regret):>{10}.3f}",flush=True)

In [ ]:
def lcb2(q1, q2, delta, rng):
    smallest = np.argmin([q1,q2])

    #first we init the variables (t=1 to t=K)
    n = np.repeat(1,K)
    regret = 0
    #the two lists below allow efficient computation of var hats
    cumul_losses = np.zeros(K)
    cumul_sq_losses = np.zeros(K)

    for t in range(K):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[t] += sampled_losses[t]
        cumul_sq_losses[t] += sampled_losses[t]**2
        regret += sampled_losses[t] - sampled_losses[smallest]
    mu_hats = np.copy(cumul_losses)
    var_hats = cumul_sq_losses/n - mu_hats**2 #var=E(x^2)-E(x)^2
    mu_deltas = mu_hats-np.sqrt(2*var_hats*np.log(1/delta)/n)-3*np.log(1/delta)/n

    for t in range(K, T):
        I_t = np.argmin(mu_deltas)

        #update variables for next iter:
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[I_t] += sampled_losses[I_t]
        cumul_sq_losses[I_t] += sampled_losses[I_t]**2

        n[I_t] += 1
        mu_hats = cumul_losses/n
        var_hats = cumul_sq_losses/n - mu_hats**2
        mu_deltas = mu_hats-np.sqrt(2*var_hats*np.log(1/delta)/n)-3*np.log(1/delta)/n

        regret += sampled_losses[I_t] - sampled_losses[smallest]

    return regret


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    for i in range(M):
        regrets[i] = lcb2(
            q1=pair[0]
            , q2=pair[1]
            , delta=1/T**2
            , rng=rng
        )
    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f}",flush=True)

print(f"{summed_avg_regret:>{14}.3f} {np.sqrt(var_summed_regret):>{10}.3f}",flush=True)

Although I'm not sure this is what is expected here we could surely also approximate this result mathematically. Since we only have two arms here, we could consider the exploration phase as a binomial experiment with for each arm $i$ with $n=\lfloor\epsilon T/2\rfloor$ and $p=q_i$. If we take $\epsilon$ to be not too small then $np$ is surely greater than $5$ or $10$ which is a commonly used heuristic for normal approximation of a binomial distribution, thus we can say our estimates $\hat q$ are normally distributed. Then for arm $i$, $\hat q_i \sim \mathcal{N}\left(\mu_i=q_i, \sigma_i=\sqrt{q_i(1-q_i)/n}\right)$. But the thing is, $q_i$ can only be between 0 and 1 while that normal distro spans the whole real axis. So
$$
\begin{align*}
E(\hat q_i) &= \int_{-\infty}^{+\infty}\hat q_i f(\hat q_i) \:d\hat q_i\\

&=\int_0^1 \hat q_i f(\hat q_i) \:d\hat q_i + \int_1^{+\infty} f(\hat q_i) \:d\hat q_i\\

&=\frac{1}{\sigma_i \sqrt{2\pi}} \left( \int_0^1 \hat q_i \exp\left( -\frac12\left(\frac{\hat q_i - \mu_i}{\sigma_i}\right)^2\right) \:d\hat q_i + \int_1^{+\infty} \exp\left( -\frac12\left(\frac{\hat q_i - \mu_i}{\sigma_i}\right)^2\right) \:d\hat q_i \right)\\
\end{align*}
$$